# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
import sys
from glob import glob

sys.path.append(os.getenv("SRC_DIR"))

from utils.logger import get_logger
_logs = get_logger(__name__)

PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
_logs.info(f"Found {len(parquet_files)} parquet files.")

2025-09-27 22:39:04,807, 1796412707.py, 15, INFO, Found 2790 parquet files.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Write your code below.

dd_px = dd.read_parquet(parquet_files).set_index("ticker")
_logs.info("Parquet files loaded into Dask DataFrame.")

dd_lags = dd_px.groupby("ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x["Close"].shift(1),
        Adj_Close_lag_1 = x["Adj Close"].shift(1)
    ),
    meta={
        'Date': 'datetime64[ns]',
        'Open': 'f8',
        'High': 'f8',
        'Low': 'f8',
        'Close': 'f8',
        'Adj Close': 'f8',
        'Volume': 'f8',
        'source': 'O',
        'Year': 'i4',
        'Close_lag_1': 'f8',
        'Adj_Close_lag_1': 'f8'
    }
)
_logs.info("Lag features calculated.")

dd_rets = dd_lags.assign(
    returns = lambda x: (x["Close"] / x["Close_lag_1"]) - 1
)
_logs.info("Returns calculated.")

dd_feat = dd_rets.assign(
    hi_lo_range = lambda x: x["High"] - x["Low"]
)
_logs.info("High-Low range feature added.")

2025-09-27 22:39:41,534, 4075193790.py, 4, INFO, Parquet files loaded into Dask DataFrame.
2025-09-27 22:39:41,543, 4075193790.py, 25, INFO, Lag features calculated.
2025-09-27 22:39:41,554, 4075193790.py, 30, INFO, Returns calculated.
2025-09-27 22:39:41,606, 4075193790.py, 35, INFO, High-Low range feature added.


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [5]:
# Write your code below.

dd_feat = dd_feat.compute()
_logs.info("Dask DataFrame computed to Pandas.")

dd_feat["returns_moving_avg_10"] = (
    dd_feat.groupby("ticker")["returns"]
    .transform(lambda x: x.rolling(10).mean())
)
_logs.info("Moving average added.")

2025-09-27 22:40:01,978, 2901646610.py, 4, INFO, Dask DataFrame computed to Pandas.
2025-09-27 22:40:02,120, 2901646610.py, 10, INFO, Moving average added.


In [6]:
_logs.info("Previewing final feature set:")
dd_feat[["Date", "Year", "Close_lag_1", "Adj_Close_lag_1", "returns", "hi_lo_range", "returns_moving_avg_10"]].head(20)

2025-09-27 22:40:02,168, 2308201549.py, 1, INFO, Previewing final feature set:


,Date,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_moving_avg_10
ticker,,,,,,,
A,1999-11-18,1999,NaN,NaN,NaN,7.153078,NaN
A,1999-11-19,1999,31.473534,27.068665,-0.082386,2.280043,NaN
A,1999-11-22,1999,28.880543,24.838577,0.089783,2.816525,NaN
A,1999-11-23,1999,31.473534,27.068665,-0.090909,2.592991,NaN
A,1999-11-24,1999,28.612303,24.607880,0.026563,1.385908,NaN
A,1999-11-26,1999,29.372318,25.261524,0.003044,0.536480,NaN
A,1999-11-29,1999,29.461731,25.338428,0.022762,1.341202,NaN
A,1999-11-30,1999,30.132332,25.915169,0.001484,1.430616,NaN
A,1999-12-01,1999,30.177038,25.953619,0.017778,1.117668,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Was it necessary to convert to Pandas to calculate the moving average return?

No—it wasn’t strictly necessary. Dask’s groupby().apply() passes each group as a Pandas DataFrame, allowing the use of .rolling().mean() inside a custom function without converting the entire dataset. This hybrid approach keeps the workflow in Dask while leveraging Pandas for operations that Dask doesn’t natively support in grouped contexts.

Reference: Dask documentation confirms that groupby().apply() delegates each group to Pandas, enabling full use of Pandas methods like .rolling() (Dask GroupBy.apply).

Would it have been better to do it in Dask? Why?

Not in this case. Although Dask is built for scalable processing, the dataset consisted of 2,790 small files (~10KB each), resulting in thousands of tiny partitions. This overwhelmed Dask’s scheduler and degraded performance. It’s a known limitation—Dask works best with fewer, larger partitions. Repartitioning would improve efficiency, but for this assignment, the pipeline remained in Dask until the final .compute() step.

Reference: Dask’s documentation on GroupBy.rolling notes that rolling operations behave differently from Pandas and operate within partitions, requiring overlap management for accurate results.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.